# Gundam NVIDIA Newton Simulation

This notebook sets up the NVIDIA Newton physics engine, loads the Gundam URDF, plays back the 'walk-forward' sample motion provided in the repository, and saves the output to a USD file on your Google Drive. You can download and open this USD file locally in NVIDIA Omniverse.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install "newton[examples]" pandas

In [ ]:
!rm -rf /content/gundam_robot
!git clone https://github.com/gundam-global-challenge/gundam_robot.git /content/gundam_robot
!sed -i 's/damping="3e2" friction="1e3"/damping="0.0" friction="0.0"/g' /content/gundam_robot/gundam_rx78_description/urdf/GGC_TestModel_rx78_20170112.urdf
# Replace package:// path with absolute path so Newton can find the meshes
!sed -i 's|package://gundam_rx78_description|/content/gundam_robot/gundam_rx78_description|g' /content/gundam_robot/gundam_rx78_description/urdf/GGC_TestModel_rx78_20170112.urdf

In [ ]:
import newton
import warp as wp
import numpy as np
import pandas as pd
from newton.solvers import SolverMuJoCo
from newton.viewer import ViewerUSD

wp.init()

# Load the sample CSV motion
csv_path = "/content/gundam_robot/gundam_rx78_control/sample/csv/walk-forward.csv"
df = pd.read_csv(csv_path)
# Clean up column names (remove leading spaces)
df.columns = df.columns.str.strip()

# Build Newton Model from URDF
print("Parsing URDF and building model...")
builder = newton.ModelBuilder()
builder.add_urdf(source="/content/gundam_robot/gundam_rx78_description/urdf/GGC_TestModel_rx78_20170112.urdf", ignore_inertial_definitions=True)
model = builder.finalize()

state = model.state()
control = model.control()
contacts = model.contacts()

# Map CSV columns to Newton joint DOFs
joint_names = builder.joint_label
csv_to_newton_idx = {}
for csv_joint in df.columns:
    if csv_joint != 'time':
        try:
            idx = joint_names.index(csv_joint)
            dof_start = builder.joint_q_start[idx]
            csv_to_newton_idx[csv_joint] = dof_start
        except ValueError:
            pass

# Initialize Solver (disable mujoco native contacts as it fails on thin meshes)
solver = SolverMuJoCo(model, use_mujoco_contacts=False)

# Setup USD Viewer
fps = 60
usd_path = "/content/drive/MyDrive/gundam_newton_motion.usda"
viewer = ViewerUSD(output_path=usd_path, fps=fps)
viewer.set_model(model)

print("Starting simulation playback...")
sim_dt = 1.0 / 200.0
render_dt = 1.0 / float(fps)
render_accum = 0.0
time_elapsed = 0.0

for idx, row in df.iterrows():
    # Read current joint states
    target_positions = state.joint_q.numpy()
    
    for csv_joint, dof_idx in csv_to_newton_idx.items():
        target_positions[dof_idx] = row[csv_joint]
    
    # Note: Proper RL would use PD control or efforts. Here we just strictly teleport the joints
    # to visualize the trajectory.
    state.joint_q = wp.array(target_positions, dtype=wp.float32)
    
    solver.step(state_in=state, state_out=state, control=control, contacts=contacts, dt=sim_dt)
    
    time_elapsed += sim_dt
    render_accum += sim_dt
    if render_accum >= render_dt:
        viewer.begin_frame(time=time_elapsed)
        viewer.log_state(state=state)
        viewer.end_frame()
        render_accum = 0.0

viewer.close()
print(f"Simulation finished. USD saved to {usd_path}")
